In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_action_relationships_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :

        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "LegalActionRelationshipsId string"
        #     ",SourceSystem string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",LegalActionRelationshipsKey string"
        #     ",LegalActionRelationshipsTypeCode string"
        #     ",LegalActionRelationshipsType string"
        #     ",LegalActionRelationshipsSubTypeCode string"
        #     ",LegalActionRelationshipsSubType string"
        #     ",LegalActionKey string"
        #     ",RelatedLegalActionKey string"
        #     ",CreationDate date"
        #     ",TerminationDate date"
        #     ",LastUpdateDate date"
        #     ",Comments string"
        #     ",PartyKey string"
        #     ",PartyName string"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalActionRelationships"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== legactrel ==================

        legactrel_extra_filters = None

        if legactrel_extra_filters is not None:
            legactrel_extra_filters.append(date_filter)
            legactrel_final_filters = " AND ".join(legactrel_extra_filters)
        else:
            legactrel_final_filters = date_filter

        legactrel_selected_fields = [
            "relationkey",
            "relationdescription",
            "primarylegalsourcekey",
            "secondarylegalsourcekey",
            "relcreationdate",
            "relterminationdate",
            "businessdate",
        ]

        legactrel = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_legactrel",
            legactrel_final_filters,
            legactrel_selected_fields,
        )

        # ================== legactcom ==================

        legactcom_extra_filters = None

        if legactcom_extra_filters is not None:
            legactcom_extra_filters.append(date_filter)
            legactcom_final_filters = " AND ".join(legactcom_extra_filters)
        else:
            legactcom_final_filters = date_filter

        legactcom_selected_fields = [
            "legalactcomid",
            "legalactid",
            "createdt",
            "comments",
            "lastupddt",
            "businessdate",
        ]

        legactcom = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_legactcom",
            legactcom_final_filters,
            legactcom_selected_fields,
        )

        # ================== legalactiontoperson ==================

        legalactiontoperson_extra_filters = None

        if legalactiontoperson_extra_filters is not None:
            legalactiontoperson_extra_filters.append(date_filter)
            legalactiontoperson_final_filters = " AND ".join(
                legalactiontoperson_extra_filters
            )
        else:
            legalactiontoperson_final_filters = date_filter

        legalactiontoperson_selected_fields = [
            "relkey",
            "relationtype",
            "legalactid",
            "cdi",
            "aliassecondary",
            "businessdate",
        ]

        legalactiontoperson = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_legalactiontoperson",
            legalactiontoperson_final_filters,
            legalactiontoperson_selected_fields,
        )

        # =============== Primary Keys - legal_action_relationships_arotron ===============

        # legactrel
        legactrel_primary_key_fields = [
            legactrel.relationkey,
            lit("LegalAction-LegalAction"),
            lit("arotron"),
        ]

        legactrel = legactrel.withColumn(
            "LegalActionRelationshipsId",
            sha2(concat_ws(";", *legactrel_primary_key_fields), 256),
        )

        # legactcom
        legactcom_primary_key_fields = [
            legactcom.legalactcomid,
            lit("LegalAction-Comments"),
            lit("arotron"),
        ]

        legactcom = legactcom.withColumn(
            "LegalActionRelationshipsId",
            sha2(concat_ws(";", *legactcom_primary_key_fields), 256),
        )

        # legalactiontoperson
        legalactiontoperson_primary_key_fields = [
            legalactiontoperson.relkey,
            lit("LegalAction-Party"),
            lit("arotron"),
        ]

        legalactiontoperson = legalactiontoperson.withColumn(
            "LegalActionRelationshipsId",
            sha2(concat_ws(";", *legalactiontoperson_primary_key_fields), 256),
        )

        # ================== Selection of Final Columns ==================

        result_df_legactrel = legactrel.select(
            legactrel.LegalActionRelationshipsId.alias("LegalActionRelationshipsId"),
            lit("arotron").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            legactrel.relationkey.cast("String").alias("LegalActionRelationshipsKey"),
            lit("LegalAction-LegalAction")
            .cast("String")
            .alias("LegalActionRelationshipsTypeCode"),
            lit("LegalAction-LegalAction")
            .cast("String")
            .alias("LegalActionRelationshipsType"),
            legactrel.relationdescription.cast("String").alias(
                "LegalActionRelationshipsSubTypeCode"
            ),
            legactrel.relationdescription.cast("String").alias(
                "LegalActionRelationshipsSubType"
            ),
            legactrel.primarylegalsourcekey.cast("String").alias("LegalActionKey"),
            legactrel.secondarylegalsourcekey.cast("String").alias(
                "RelatedLegalActionKey"
            ),
            legactrel.relcreationdate.cast("date").alias("CreationDate"),
            legactrel.relterminationdate.cast("date").alias("TerminationDate"),
            lit(None).cast("date").alias("LastUpdateDate"),
            lit(None).cast("String").alias("Comments"),
            lit(None).cast("String").alias("PartyKey"),
            lit(None).cast("String").alias("PartyName"),
            legactrel.businessdate.cast("date").alias("BusinessDate"),
        )

        result_df_legactcom = legactcom.select(
            legactcom.LegalActionRelationshipsId.alias("LegalActionRelationshipsId"),
            lit("arotron").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            legactcom.legalactcomid.cast("String").alias("LegalActionRelationshipsKey"),
            lit("LegalAction-Comments")
            .cast("String")
            .alias("LegalActionRelationshipsTypeCode"),
            lit("LegalAction-Comments")
            .cast("String")
            .alias("LegalActionRelationshipsType"),
            lit(None).cast("String").alias("LegalActionRelationshipsSubTypeCode"),
            lit(None).cast("String").alias("LegalActionRelationshipsSubType"),
            legactcom.legalactid.cast("String").alias("LegalActionKey"),
            lit(None).cast("String").alias("RelatedLegalActionKey"),
            legactcom.createdt.cast("date").alias("CreationDate"),
            lit(None).cast("date").alias("TerminationDate"),
            legactcom.lastupddt.cast("date").alias("LastUpdateDate"),
            legactcom.comments.cast("String").alias("Comments"),
            lit(None).cast("String").alias("PartyKey"),
            lit(None).cast("String").alias("PartyName"),
            legactcom.businessdate.cast("date").alias("BusinessDate"),
        )

        result_df_legalactiontoperson = legalactiontoperson.select(
            legalactiontoperson.LegalActionRelationshipsId.alias(
                "LegalActionRelationshipsId"
            ),
            lit("arotron").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            legalactiontoperson.relkey.cast("String").alias(
                "LegalActionRelationshipsKey"
            ),
            lit("LegalAction-Party")
            .cast("String")
            .alias("LegalActionRelationshipsTypeCode"),
            lit("LegalAction-Party")
            .cast("String")
            .alias("LegalActionRelationshipsType"),
            legalactiontoperson.relationtype.cast("String").alias(
                "LegalActionRelationshipsSubTypeCode"
            ),
            legalactiontoperson.relationtype.cast("String").alias(
                "LegalActionRelationshipsSubType"
            ),
            legalactiontoperson.legalactid.cast("String").alias("LegalActionKey"),
            lit(None).cast("String").alias("RelatedLegalActionKey"),
            lit(None).cast("date").alias("CreationDate"),
            lit(None).cast("date").alias("TerminationDate"),
            lit(None).cast("date").alias("LastUpdateDate"),
            lit(None).cast("String").alias("Comments"),
            legalactiontoperson.cdi.cast("String").alias("PartyKey"),
            legalactiontoperson.aliassecondary.cast("String").alias("PartyName"),
            legalactiontoperson.businessdate.cast("date").alias("BusinessDate"),
        )

        legal_action_relationships_arotron = result_df_legactrel.union(
            result_df_legactcom
        ).union(result_df_legalactiontoperson)

        # ================== Append ==================

        legal_action_relationships_arotron.write.format("delta").clusterBy(
            "BusinessDate"
        ).mode("append").saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_legal_action_relationships_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)